### Hugging Face
- 이미지캡셔닝   :  Salesforce/blip-image-captioning-base
- 텍스트 임베딩  :  sentence-transformers/all-MiniLM-L6-v2
- 텍스트 생성    : distilbert/distilgpt2  (경량 모델)

In [1]:
import os
import fitz
import chromadb
from PIL import Image
from transformers import pipeline,BlipProcessor, BlipForConditionalGeneration
from chromadb.utils import embedding_functions

In [2]:
# BLIP 모델 로드
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [3]:
# 임베딩 모델 로드
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# vectorDB 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db_hf')
collection = chroma_client.get_or_create_collection(name='multimocal_rag_hf',embedding_function=hf_ef)

### 데이터추출 및 오픈소스 임베딩
- PDF 텍스트와 추출된 이미지를 BLIP 모델로 갭셔닝한 뒤, Sentence Transformers임베딩을 사용해서 로컬 vectorDB에 저장

In [5]:
ptf_path = 'sample_paper.pdf'
if collection.count() == 0:
    doc = fitz.open(ptf_path)
    documents,metadatas,ids = [],[],[]
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추가
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metadatas.append({'type':'text','page':page_num+1})
            ids.append(f'hf_text_page_{page_num+1}')
        # 이미지 캡셔닝
        image_lists = page.get_images(full=True)
        if image_lists:
            xref = image_lists[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            image_filename = f'ht_temp_img.{ext}'
            with open(image_filename, 'wb') as f:
                f.write(image_bytes)
            
            try:
                raw_image = Image.open(image_filename).convert('RGB')
                inputs = processor(raw_image, return_tensors='pt')
                out = model.generate(**inputs,max_new_tokens=50)
                caption = processor.decode(out[0],skip_special_tokens=True)

                documents.append(caption)
                metadatas.append({'type':'image_summary', 'page':page_num+1})
                ids.append(f'hf_image_summary_page_{page_num+1}')
                print(f'페이지 {page_num+1} 이미지 캡셔닝 완료 : {caption}')
            except Exception as e:
                print(f'페이지 {page_num+1} 이미지 캡셔닝 실패 : {e}')
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
else:
    print(collection.count())


15


### RAG 검색 및 LLM을 통한 답변 생성

In [ ]:
query = 'what is transformer architecture?'
results = collection.query(
    query_texts=[query],
    n_results=1
)
retrieved_docs = results['documents'][0]
retrieved_metas = results['metadatas'][0]
print(f'검색된 문서 수 : {len(retrieved_docs)}')
for meta in retrieved_metas:
    print(f" - [출처:페이지{meta['page']}, 타입] : {meta['type']}")
context = '\n\n--\n\n'.join(retrieved_docs) 
print('\n오픈소스 답변 생성중\n')
generator = pipeline('text-generation', model='distilbert/distilgpt2')
prompt = f'Context:{context[:400]}\n\nQuestion:{query}\n\nAnswer:'
response = generator(prompt,max_new_tokens=100, num_return_sequences=1)
print('최종답변')
print(response[0]['generated_text'].replace(prompt, "").strip())

{'ids': [['hf_text_page_3']],
 'embeddings': None,
 'documents': [['Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1\nEncoder and Decoder Stacks\nEncoder:\nThe encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-\nwise fully connected feed-forward network. We employ a residual connection [11] around each of\nthe two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer\nitself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimension dmodel = 